In [ ]:
import os
import tkinter as tk
from tkinter import filedialog
import numpy as np
import psycopg2
from pgvector.psycopg2 import register_vector
from deepface import DeepFace

DB_CONFIG = {
    "dbname": "FACEPROJECT",
    "user": "postgres",
    "password": os.environ.get("DB_PASSWORD", ""), 
    "host": "localhost",
    "port": 5432
}

MODEL_NAME = "ArcFace"           
DETECTOR_BACKEND = "retinaface"   


def pick_image():
    """Opens a system file dialog to select the target photo."""
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    path = filedialog.askopenfilename(
        title="Select Missing Person's Photo",
        filetypes=[("Image Files", "*.jpg *.jpeg *.png *.webp *.bmp"), ("All Files", "*.*")]
    )
    root.destroy()
    return path


def main():
    print("=" * 45)
    print("      ENROLL MISSING PERSON INTO FACEPROJECT ")
    print("=" * 45)

    name = input("Enter Full Name: ").strip()
    if not name:
        print("[!] Name cannot be empty.")
        return

    age_input = input("Enter Age (or press Enter to skip): ").strip()
    age = int(age_input) if age_input.isdigit() else None

    gender = input("Enter Gender (e.g., Male/Female/Other): ").strip()
    clothes = input("Clothes Description (e.g., Black hoodie, blue jeans): ").strip()
    last_seen = input("Last Seen Location (e.g., Central Station): ").strip()

    print("\n[*] Please choose the clear photo of the person in the popup window...")
    photo_path = pick_image()

    if not photo_path:
        print("[!] No image was selected. Aborting enrollment.")
        return

    print(f"[+] Selected image: {photo_path}")
    print("[*] Detecting face and generating 512-d ArcFace embedding...")

    try:
        # DeepFace extracts, aligns, and computes the 512-d ArcFace vector
        reps = DeepFace.represent(
            img_path=photo_path,
            model_name=MODEL_NAME,
            detector_backend=DETECTOR_BACKEND,
            enforce_detection=True
        )
        raw_embedding = reps[0]["embedding"]
        embedding = np.array(raw_embedding, dtype=np.float32)

        # Normalize the embedding to unit length for optimal cosine distance performance
        norm = np.linalg.norm(embedding)
        if norm > 0:
            embedding = embedding / norm

        print(f"[+] Embedding computed successfully (Vector dimension: {len(embedding)})")

    except Exception as e:
        print(f"[!] Face detection/embedding generation failed: {e}")
        return

    # Connect to PostgreSQL and insert the record
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        register_vector(conn)
        cur = conn.cursor()

        insert_query = """
            INSERT INTO missing_persons (
                name, age, gender, photo_path, 
                clothes_description, last_seen_location, face_embedding
            ) VALUES (%s, %s, %s, %s, %s, %s, %s)
            RETURNING id, created_at;
        """

        cur.execute(
            insert_query,
            (name, age, gender, photo_path, clothes, last_seen, embedding)
        )
        record_id, created_at = cur.fetchone()
        conn.commit()

        print("\n" + "=" * 45)
        print(f"[+] RECORD ENROLLED SUCCESSFULLY!")
        print(f"    Assigned Database ID : {record_id}")
        print(f"    Name                 : {name}")
        print(f"    Timestamp            : {created_at}")
        print("=" * 45)

        # Verify the record count
        cur.execute("SELECT COUNT(*) FROM missing_persons;")
        total = cur.fetchone()[0]
        print(f"[*] Total persons currently in FACEPROJECT: {total}")

        cur.close()
        conn.close()

    except Exception as e:
        print(f"[!] Database insertion error: {e}")


if __name__ == "__main__":
    main()

      ENROLL MISSING PERSON INTO FACEPROJECT 
